# 05 · Fit the fuser, calibrate, run the acceptance gate

This stage produces the two artefacts the deployed app reads — `fusion_a.json`
and `fusion_b.json` — and the honest evaluation report.

**Calibration is not optional (A6).** A reported "82% AI" must correspond to
an observed 82% positive rate within 5 points. Uncalibrated confidence is the
single most common failure of tools in this category, and it is what makes a
score usable as evidence rather than as a vibe.

**A4 is a release blocker.** Detectors are known to over-flag non-native
English writers. If the ESL false-positive rate exceeds 5%, this build does
not ship, regardless of how good every other number looks.

---

> ### This stage needs evaluation splits that do not exist yet
>
> It requires seven held-out `eval_*.parquet` files — RAID-test, M4, an
> academic essay set, an ESL/ELL corpus, and calibration splits — each already
> scored into the four signal columns (`classifier`, `features`,
> `binoculars_ratio`, `burstiness`).
>
> Building them is real work: score each held-out corpus with the ONNX models
> exported in stage 4 plus `api/_lib/stats.py`, and save the four columns
> alongside the label.
>
> **The cells below detect the missing files and skip cleanly**, so a
> "Save & Run All" still finishes green with stages 1–4 complete. Until this
> stage runs you have a working trained detector, but no calibration and no
> A4 number — so do not publish accuracy claims yet.


In [ ]:
# Setup. Run once per session — everything below depends on it.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow google-api-python-client google-auth

import sys, os
from pathlib import Path

# Must be set before torch is imported anywhere — PyTorch reads it when CUDA
# first initialises. Reduces the fragmentation that turns "enough memory" into
# an out-of-memory error hours into a run.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Change this if you forked the repo. Public repo => no token needed.
GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"

# Either attached as a Kaggle Dataset named `ai-detector-repo`, or cloned.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

# parents=True so this also works off-Kaggle (Colab, a local box) after
# pointing WORK somewhere that exists.
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("work:", WORK)
assert (REPO / "training" / "lib").is_dir(), "repo not found — check GIT_URL"


In [ ]:
# ---------------------------------------------------------------------------
# Run configuration. Read this cell before starting anything else.
# ---------------------------------------------------------------------------
#
# Kaggle gives 30 GPU-hours a week. A full run is 8-16 of them, so you cannot
# afford to discover a bug at hour six. Leave SMOKE_TEST = True for the first
# pass: it runs the entire pipeline end to end in well under an hour on a
# tiny sample. If stage 4 completes, the chain works. Then set it False and
# run for real.

SMOKE_TEST = True

if SMOKE_TEST:
    SAMPLE_ROWS = 5_000     # rows per dataset
    EPOCHS = 1
else:
    SAMPLE_ROWS = 400_000   # PRD 12.2
    EPOCHS = 3

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}: "
      f"{SAMPLE_ROWS:,} rows/dataset, {EPOCHS} epoch(s)")
if not SMOKE_TEST:
    print("Expect ~1-2 h for stage 1, then 4-8 h per model. Use "
          "Save Version -> Save & Run All so a browser disconnect cannot kill it.")


In [ ]:
# ---------------------------------------------------------------------------
# Google Drive — the one place work survives the session ending.
# ---------------------------------------------------------------------------
#
# Kaggle wipes /kaggle/working when a session ends or times out. Everything
# expensive — the dataset, the newest checkpoint, the final model — is written
# once locally and uploaded here. There is no second copy and no second
# cadence; the file you see on disk is the file that gets uploaded.
#
# One-time setup:
#   1. console.cloud.google.com -> new project -> enable the Drive API
#   2. Create a service account, then create a JSON key for it
#   3. In Drive, make a folder and share it with the service account's
#      client_email (found in the JSON) as Editor.
#      A service account has its own Drive with ZERO quota, so it can only
#      write into a folder you have shared with it, where the bytes count
#      against your quota. This is the step people miss, and it fails with a
#      confusing quota error rather than a permission error.
#   4. The folder id is the last part of
#      drive.google.com/drive/folders/<THIS_PART>
#   5. Upload the JSON key to Kaggle as a PRIVATE dataset, then point at it.
#      Keep it private: that key can write to the folder you shared.

DRIVE_FOLDER_ID = ""   # e.g. "1AbC2dEfGhIjKlMnOpQrStUvWxYz"
DRIVE_KEY_PATH  = ""   # e.g. "/kaggle/input/gdrive-key/service_account.json"

import os

if DRIVE_FOLDER_ID and DRIVE_KEY_PATH:
    os.environ["DRIVE_FOLDER_ID"] = DRIVE_FOLDER_ID
    os.environ["DRIVE_SERVICE_ACCOUNT_JSON"] = DRIVE_KEY_PATH

from lib.store import build_store
STORE = build_store()

if STORE.__class__.__name__ == "NullStore":
    print("")
    print("Nothing will survive this session ending. Fine for a smoke test;")
    print("fill in the two values above before starting a real run.")
else:
    # Prove the credentials work now, rather than discovering they do not
    # eight hours in when the first checkpoint tries to upload.
    from pathlib import Path
    probe = Path("/kaggle/working/.drive_probe")
    probe.write_text("ok")
    if STORE.push(probe, "_probe.txt") and STORE.pull("_probe.txt", probe):
        print("Drive write + read verified.")
    else:
        print("Drive is NOT working. Check that the folder is shared with the")
        print("service account's client_email as Editor.")
    probe.unlink(missing_ok=True)


In [ ]:
import numpy as np, pandas as pd, json
from lib.calibrate import (_logit, _sigmoid, calibration_error, fit_fusion,
                           fit_isotonic, run_acceptance_gate, write_fusion_config)

REQUIRED_SPLITS = ["calibration_a", "calibration_b", "in_domain", "m4",
                   "essays", "esl", "humanized"]
missing = [n for n in REQUIRED_SPLITS if not (DATA / f"eval_{n}.parquet").is_file()]
EVAL_READY = not missing

if EVAL_READY:
    print("All evaluation splits present. Stage 5 will run.")
else:
    print("Stage 5 SKIPPED — these files are missing from", DATA)
    for name in missing:
        print(f"  eval_{name}.parquet")
    print("\nStages 1-4 are unaffected. See the note above for how to build these.")

def load_split(name):
    """Return (signals[N,4], labels[N]) in the fuser's signal order."""
    frame = pd.read_parquet(DATA / f"eval_{name}.parquet")
    signals = frame[["classifier", "features", "binoculars_ratio",
                     "burstiness"]].to_numpy(np.float64)
    return signals, frame["label"].to_numpy(np.float64)


In [ ]:
# Fit the fuser on a calibration split — never on anything used for training
# or for the final report.
if EVAL_READY:
    for tag in ("a", "b"):
        signals, labels = load_split(f"calibration_{tag}")
        weights, bias = fit_fusion(signals, labels)
        print(f"Model {tag.upper()} weights:",
              dict(zip(["classifier", "features", "binoculars", "burstiness"],
                       weights.round(3))), f"bias {bias:.3f}")

        raw = _sigmoid(_logit(signals) @ weights + bias)
        before = calibration_error(raw, labels)
        x, y = fit_isotonic(raw, labels)
        after = calibration_error(np.interp(raw, x, y), labels)
        print(f"  worst decile gap: {before['max_gap_points']:.1f} -> "
              f"{after['max_gap_points']:.1f} points")

        write_fusion_config(MODELS / f"fusion_{tag}.json", weights, bias, x, y,
                            version=f"{tag}-1.0.0")
else:
    print("skipped")


In [ ]:
# The acceptance gate (PRD 14). Every number here goes in the published report.
if EVAL_READY:
    def fused(tag, split):
        cfg = json.loads((MODELS / f"fusion_{tag}.json").read_text())
        signals, labels = load_split(split)
        raw = _sigmoid(_logit(signals) @ np.asarray(cfg["weights"]) + cfg["bias"])
        return np.interp(raw, cfg["calibration_x"], cfg["calibration_y"]), labels

    manifest = json.loads((MODELS / "manifest.json").read_text())

    report = run_acceptance_gate(
        in_domain=fused("a", "in_domain"),
        cross_domain=fused("a", "m4"),
        essays=fused("a", "essays"),
        esl=fused("a", "esl"),
        humanized=fused("a", "humanized"),
        calibrated=fused("a", "in_domain"),
        onnx_max_logit_delta=manifest["parity"]["a"]["max_logit_delta"],
        bundle_sizes_mb=manifest["bundle_sizes_mb"],
    )
    print(report.summary())
    report.write(MODELS / "eval_report.json")
else:
    print("skipped")


In [ ]:
# A4 is a release blocker. This cell is meant to stop the pipeline.
if EVAL_READY:
    blockers = report.blockers_failed
    if blockers:
        raise SystemExit(
            "DO NOT SHIP. Blocking criteria failed: " +
            ", ".join(c.id for c in blockers) +
            "\n\nA4 exists because detectors are known to over-flag non-native "
            "English writers. Shipping a tool that penalises ESL authors is a real "
            "harm, not a metric regression. Retrain with more ESL data in the "
            "negative class and a lower alpha in the partial-AUROC loss."
        )
    print("No blocking criteria failed.")
else:
    print("skipped")


In [ ]:
# A7 and A8 come from the humanizer, not the detector. Run them against the
# deployed app once the models are live:
#
#   A7  surrogate score > 0.9 falls below 0.3 within 3 passes, >= 90% of docs
#   A8  meaning similarity retained >= 0.85
#
#   python scripts/eval_humanizer.py --url https://your-app.vercel.app \
#       --input eval/ai_documents.jsonl --output models/eval_report.json
print("Pipeline finished.")
